### Study Region & Data

Region: Northwest Atlantic Gulf Stream

Lon: -81 to -56

Lat: 29 to 44

Date Snapshot Used: 2025-03-12

In [53]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

LON_RANGE = [-81, -56]
LAT_RANGE = [29, 44]
DATA_DIR = Path(".")


In [54]:
# Single-day snapshot: 2025-03-12
ssh = xr.open_dataset(DATA_DIR / "dt_global_allsat_phy_l4_20250312_20250826.nc")
adt = ssh["adt"].squeeze()

pace = xr.open_dataset(DATA_DIR / "PACE_OCI.20250312.L3m.DAY.RRS.V3_1.Rrs.4km.nc")
wl = pace["wavelength"].values
rrs_domain_mean = np.nanmean(pace["Rrs"].values, axis=(0, 1))


In [ ]:
proj = ccrs.PlateCarree()

fig = plt.figure(figsize=(10, 8.5), dpi=300)
gs = fig.add_gridspec(2, 1, height_ratios=[1, 1], hspace=0.25)
ax1 = fig.add_subplot(gs[0], projection=proj)
ax2 = fig.add_subplot(gs[1])

# Panel A: ADT
ax1.set_extent([*LON_RANGE, *LAT_RANGE], crs=proj)
vlim = np.ceil(max(abs(float(adt.min())), abs(float(adt.max()))) * 4) / 4
cf = ax1.contourf(
    adt.longitude, adt.latitude, adt.values,
    levels=np.linspace(-vlim, vlim, 40), cmap="RdBu_r",
    transform=proj, zorder=0, extend="both",
)
ax1.add_feature(cfeature.NaturalEarthFeature("physical", "land", "10m",
                facecolor="#e8e8e8", edgecolor="none"), zorder=1)
ax1.add_feature(cfeature.NaturalEarthFeature("physical", "coastline", "10m",
                facecolor="none", edgecolor="0.3", linewidth=0.6), zorder=2)
for name, lon, lat, sz, rot in [("Gulf Stream", -70, 38.5, 13, 0),
                                 ("Sargasso Sea", -62, 30.5, 12, 0)]:
    ax1.text(lon, lat, name, fontsize=sz, fontstyle="italic", color="0.15",
             ha="center", va="center", rotation=rot, transform=proj, zorder=5)
gl1 = ax1.gridlines(draw_labels=True, linewidth=0.3, color="0.5", alpha=0.5, zorder=4)
gl1.top_labels = gl1.right_labels = False
gl1.xlabel_style = {"size": 11}
gl1.ylabel_style = {"size": 11}
cbar1 = fig.colorbar(cf, ax=ax1, orientation="vertical", shrink=0.75, pad=0.02, aspect=25)
cbar1.set_ticks(np.arange(-vlim, vlim + 0.125, 0.5))
cbar1.set_label("(m)", fontsize=11)
cbar1.ax.tick_params(labelsize=10)
ax1.set_title("(a) Absolute Dynamic Topography", fontsize=13, pad=10)

# Inset
inset_ax = inset_axes(ax1, width="22%", height="30%", loc="upper left",
                      axes_class=cartopy.mpl.geoaxes.GeoAxes,
                      axes_kwargs={"projection": proj}, borderpad=1.5)
inset_ax.set_extent([-100, -30, 10, 55], crs=proj)
inset_ax.add_feature(cfeature.LAND, facecolor="#e8e8e8", edgecolor="0.5", linewidth=0.3)
inset_ax.add_feature(cfeature.OCEAN, facecolor="#dbe9f6")
inset_ax.add_patch(mpatches.Rectangle(
    (LON_RANGE[0], LAT_RANGE[0]),
    LON_RANGE[1] - LON_RANGE[0], LAT_RANGE[1] - LAT_RANGE[0],
    linewidth=1.5, edgecolor="red", facecolor="red", alpha=0.25, transform=proj,
))

# Panel B: Rrs
ax2.plot(wl, rrs_domain_mean, color="#006d6f", lw=2)
ax2.set_xlabel("Wavelength (nm)", fontsize=12)
ax2.set_ylabel(r"$R_{\mathrm{rs}}$ (sr$^{-1}$)", fontsize=12)
ax2.set_xlim(345, 720)
ax2.set_ylim(bottom=0)
ax2.tick_params(labelsize=11)
ax2.set_title(r"(b) Remote Sensing Reflectance ($R_{\mathrm{rs}}$)", fontsize=13, pad=10)
ax2.spines[["top", "right"]].set_visible(False)

# Align ax2 to span full width of ax1 + colorbar
fig.canvas.draw()
pos1 = ax1.get_position()
cbar_right = cbar1.ax.get_position().x1
pos2 = ax2.get_position()
ax2.set_position([pos1.x0, pos2.y0, cbar_right - pos1.x0, pos1.height])

# Separator
pos2_new = ax2.get_position()
mid_y = (pos1.y0 + pos2_new.y1) / 2
fig.add_artist(plt.Line2D([pos1.x0, cbar_right], [mid_y, mid_y],
               transform=fig.transFigure, color="0.75", linewidth=0.8))

fig.savefig("study_region_map.png", dpi=300, bbox_inches="tight")
plt.show()